In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers
!pip install -q accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 66.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 55.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 4.8 MB/s eta 0:00:00


In [1]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM/

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

torch.set_default_dtype(torch.float32)

from master_init import *
from DSG import *

In [3]:
config = {
    "device" : "cuda:0",
    "device_ids" : [0]
}
device = config["device"]
device_ids = config["device_ids"]

# model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device)
model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device, dtype=torch.float32)

In [4]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["Brain2Image"],
    bsz=[1],
    dev_bsz=[1]
)

## Code

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def train(args_dict):
    dataloader = args_dict["dataloader"]
    model = args_dict["model"]
    optimizer = args_dict["optimizer"]
    criterion = args_dict["criterion"]
    device = args_dict["device"] if "device" in args_dict else "cuda"
    device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
    staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None
    symmetric_KL = lambda a, b : 0.5 * (criterion(F.log_softmax(a, dim=1), F.softmax(b, dim=1)) + criterion(F.log_softmax(b, dim=1), F.softmax(a, dim=1)))
    real_bsz = args_dict["real_bsz"] if "real_bsz" in args_dict else 64

    if staging_device==None:
        staging_device = f"cuda:{device_ids[0]}" if device_ids == None else "cuda"
    results = {}
    for phase in ['train', 'dev']:
        if phase == 'train':
            model.train()    # Set model to training mode
        else:
            dataloader[phase]
            model.eval()     # Set model to evaluate mode

        running_loss = 0.0
        tot_cnt = 0

        # Iterate over data.
        current_data = dataloader[phase].load_data()
        while not current_data["reset"]:
            lst_targets = []
            lst_outputs = []
            for i in range(real_bsz):
                eegs = current_data["data"]
                if current_data["reset"]:
                    break
                args_dict = {
                    "input_data_batch" : eegs.to(device, dtype=torch.float32),
                    "pool_result" : True
                    }

                model.zero_grad()
                output = model(
                    mode="PRETRAIN-EEG-IMG-CLIP-MATCHING",
                    args_dict=args_dict,
                    staging_device=staging_device,
                )

                lst_outputs.append(output)
                lst_targets.append(current_data["target"][0])
                current_data = dataloader[phase].load_data()

            target_embed = torch.cat(lst_targets, dim=0).to(dtype=torch.float32)
            output = torch.cat(lst_outputs, dim=0).to(dtype=torch.float32)

            optimizer.zero_grad()

            target_embeds_pooled = target_embed
            target_pairwise_embeds = torch.mm(target_embeds_pooled, target_embeds_pooled.T)
            output = torch.mm(output, target_embeds_pooled.T)

            loss = symmetric_KL(output, target_pairwise_embeds)

            # Backward + Optimize only if in training phase
            if phase == 'train':
                if device_ids == None:
                    loss.backward()
                    optimizer.step()
                else:
                    loss.mean().backward()
                    optimizer.step()

            # Compute stats
            if device_ids == None or len(device_ids) == 1:
                running_loss += loss.item() * eegs.size()[0]
            else:
                running_loss += loss.mean().item() * eegs.size()[0]
            tot_cnt += eegs.size()[0]

        epoch_loss = running_loss / tot_cnt

        results[f"{phase}_loss"] = epoch_loss
    results["model"] = model
    return results

In [7]:
args_dict = {
    "dataloader" : dataset_dict["Brain2Image"],
    "model" : model,
    "optimizer" : optim.Adam(model.parameters(), lr=1e-5),
    "criterion" : nn.KLDivLoss(reduction="batchmean"),
    "device" : device,
    "device_ids" : device_ids,
    "staging_device" : "cuda:0",
    "real_bsz" : 8
}

In [ ]:
results = train(args_dict)

In [30]:
# dataloader = args_dict["dataloader"]
# model = args_dict["model"]
# optimizer = args_dict["optimizer"]
# criterion = args_dict["criterion"]
# device = args_dict["device"] if "device" in args_dict else "cuda"
# device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
# staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None
# symmetric_KL = lambda a, b : 0.5 * (criterion(F.log_softmax(a, dim=1), F.softmax(b, dim=1)) + criterion(F.log_softmax(b, dim=1), F.softmax(a, dim=1)))
# real_bsz = args_dict["real_bsz"] if "real_bsz" in args_dict else 64

In [8]:
# phase="train"
# running_loss = 0.0
# tot_cnt = 0

# # Iterate over data.
# current_data = dataloader[phase].load_data()
# while not current_data["reset"]:
#     eegs = current_data["data"]

#     optimizer.zero_grad()

#     args_dict = {
#         "input_data_batch" : eegs.to(device, dtype=torch.float32),
#         "pool_result" : True
#         }

#     output = model(
#         mode="PRETRAIN-EEG-IMG-CLIP-MATCHING",
#         args_dict=args_dict,
#         staging_device=staging_device,
#     )

#     target_embed = current_data["target"].to(torch.float32)
#     target_embeds_pooled = torch.mean(target_embed, dim=1)
#     target_pairwise_embeds = torch.mm(target_embeds_pooled, target_embeds_pooled.T)

#     output = torch.mean(output, dim=1).to(torch.float32)
#     output = torch.mm(output, target_embeds_pooled.T)

#     loss = symmetric_KL(output, target_pairwise_embeds)

#     # Backward + Optimize only if in training phase
#     if phase == 'train':
#         if device_ids == None:
#             loss.backward()
#             optimizer.step()
#         else:
#             loss.mean().backward()
#             optimizer.step()

#     # Compute stats
#     if device_ids == None or len(device_ids) == 1:
#         running_loss += loss.item() * eegs.size()[0]
#     else:
#         running_loss += loss.mean().item() * eegs.size()[0]
#     tot_cnt += eegs.size()[0]
#     current_data = dataloader[phase].load_data()

# epoch_loss = running_loss / tot_cnt

AttributeError: ignored

In [11]:
# phase = "train"
# current_data = dataloader[phase].load_data()
# lst_targets = []
# lst_outputs = []
# for i in range(real_bsz):
#     eegs = current_data["data"]
#     args_dict = {
#         "input_data_batch" : eegs.to(device, dtype=torch.float32),
#         "pool_result" : True
#         }

#     model.zero_grad()
#     output = model(
#         mode="PRETRAIN-EEG-IMG-CLIP-MATCHING",
#         args_dict=args_dict,
#         staging_device=staging_device,
#     )

#     lst_outputs.append(output)
#     lst_targets.append(current_data["target"][0])
#     current_data = dataloader[phase].load_data()

# target_embed = torch.cat(lst_targets, dim=0).to(dtype=torch.float32)
# output = torch.cat(lst_outputs, dim=0).to(dtype=torch.float32)

In [22]:
# target_embeds_pooled = target_embed
# target_pairwise_embeds = torch.mm(target_embeds_pooled, target_embeds_pooled.T)
# output = torch.mm(output, target_embeds_pooled.T)

In [25]:
# loss = symmetric_KL(output, target_pairwise_embeds)

In [26]:
# loss.backward()

In [27]:
# optimizer.step()